## DOUAE

# BESOIN 3 : ARCHITECTURE DE CLASSIFICATION ET SAUVEGARDE DES IMAGES PNG

### 1. Importation des bibliothèques et configuration

pandas/numpy pour manipuler les données, joblib pour sauvegarder le scaler et le modèle, matplotlib/seaborn pour les graphiques, et le nécessaire scikit-learn pour le split, GridSearchCV et les métriques.

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")

## 2. Préparation et nettoyage des données

Cible : `implantation`.

On a discuté de ça en groupe : à la base on voulait rester sur des variables 100% techniques (puissance, prises...) pour éviter tout raisonnement circulaire, mais pour pousser l'accuracy plus loin on a décidé d'ajouter la latitude/longitude et les variables de paiement, en sachant que ça pose deux soucis qu'il faut garder en tête :
- `paiement_acte/cb/autre` sont plutôt des conséquences du lieu (un parking privé n'a pas le même mode de paiement qu'une rue) que des vraies causes. On l'avait déjà identifié et viré dans la version précédente, on le remet ici en connaissance de cause.
- la lat/long avec un split aléatoire pose un problème de fuite : une station a en moyenne 3.3 points de charge, donc la même station peut se retrouver à la fois en train et en test. On a testé : ça donne 0.95 avec le split classique contre seulement 0.81 avec un split qui regroupe les lignes par station (donc sans fuite possible). On garde le split aléatoire ici quand même, mais l'accuracy de la section 6 est probablement optimiste à cause de ça.

Variables gardées :

Numériques : `puissance`, `nb_pdc`, `latitude`, `longitude`

Booléennes : `gratuit`, `deux_roues`, `prise_ccs`, `prise_type2`, `prise_chademo`, `prise_ef`, `paiement_acte`, `paiement_cb`, `paiement_autre`

Catégorielle (encodée en OneHot, voir 4.2) : `type_tarif`

On a viré `prise_autre` : seulement 2.8% de présence dans le dataset, c'est trop peu pour que ça serve à quelque chose (et son importance mesurée dans la version précédente était déjà quasi nulle).

On a aussi regardé la corrélation entre toutes les variables candidates (section 3.1) avant de les garder : rien ne dépasse 0.9, donc pas de doublon flagrant à enlever. La paire la plus corrélée c'est `prise_ccs`/`prise_type2` (-0.82), ce qui est normal puisqu'une borne a rarement les deux en même temps.

`tarification_brute` est inutilisable : 89% de valeurs manquantes et le reste c'est du texte libre (des URLs, des paragraphes...). On a pris `type_tarif` à la place, qui est propre (5 catégories, 0% de NA).

Pour le nettoyage on a juste fait un `dropna` sur les colonnes qu'on garde plus la cible.

In [2]:
df = pd.read_csv("export_IA.csv", low_memory=False)

colonne_cible = 'implantation'

colonnes_numeriques = ['puissance', 'nb_pdc', 'latitude', 'longitude']
colonnes_booleennes = ['gratuit', 'deux_roues', 'prise_ccs', 'prise_type2', 'prise_chademo', 'prise_ef',
                        'paiement_acte', 'paiement_cb', 'paiement_autre']
colonne_categorielle = 'type_tarif'

colonnes_features = colonnes_numeriques + colonnes_booleennes + [colonne_categorielle]

df_propre = df.dropna(subset=colonnes_features + [colonne_cible]).copy()
print(f"Lignes disponibles après nettoyage : {len(df_propre)} / {len(df)}")


Lignes disponibles après nettoyage : 104282 / 138934


### 2.1 Encodage des variables catégorielles

Pour les booléens on a fait un mapping à la main (`TRUE`/`FALSE`/`OUI`/`NON`/`0`/`1` vers `0`/`1`), et tout ce qu'on reconnaît pas est mis à 0.

Pour `type_tarif` on a hésité entre plusieurs façons de l'encoder :
- Un `LabelEncoder` (ou `OrdinalEncoder`) aurait transformé les 5 catégories en 0, 1, 2, 3, 4 — mais ça impose un ordre qui n'existe pas (`kwh` n'est pas "plus grand" que `composite`), et un modèle linéaire comme la régression logistique aurait pu interpréter ça comme une vraie relation d'ordre. Random Forest s'en fiche un peu plus, mais comme on compare aussi avec la régression logistique, autant éviter le problème.
- Supprimer la variable, on a écarté direct, parce qu'on a vu en analysant la corrélation qu'elle sépare bien certaines classes.
- Un encodage par fréquence ou par moyenne de la cible (target encoding), on a écarté aussi : pour seulement 5 catégories c'est overkill, et en plus mal fait ça peut carrément fuiter l'info de la cible.
- Donc on a pris OneHotEncoder : une colonne binaire par catégorie, pas d'ordre supposé. C'est la méthode classique pour du nominal avec peu de catégories.

In [3]:
for col in colonnes_booleennes:
    df_propre[col] = df_propre[col].astype(str).str.upper()
    df_propre[col] = df_propre[col].map({'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0, 'OUI': 1, 'NON': 0}).fillna(0).astype(int)


## 3. Analyse Exploratoire des Données (Visualisations)

Deux graphiques pour vérifier nos intuitions avant de foncer dans le modèle :
- Boxplot de la puissance par implantation, pour voir si ça varie vraiment selon le lieu.
- Barplot de la proportion de prise CCS par implantation, parce qu'on s'attendait à ce que ça varie beaucoup entre une station dédiée et un parking résidentiel — le graphique confirme l'intuition.

In [4]:
print("Enregistrement des graphiques de justification...")

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_propre, x=colonne_cible, y='puissance', hue=colonne_cible, legend=False, palette='Set2')
plt.title("Justification de la variable 'puissance' selon l'Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("output/justification_puissance.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 5))
sns.barplot(data=df_propre, x=colonne_cible, y='prise_ccs', errorbar=None, hue=colonne_cible, legend=False, palette='Blues_r')
plt.title("Proportion de présence de Prises CCS par Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("output/proportion_prise_ccs.png", dpi=300)
plt.close()

Enregistrement des graphiques de justification...


## 3.1 Matrice de corrélation et audit des variables disponibles

On vérifie ici qu'on n'a pas deux variables qui racontent la même chose (la catégorie `type_tarif` n'est pas dans cette matrice, ça n'a pas de sens de calculer une corrélation sur du nominal avant de l'encoder en OneHot).

On considère qu'au-delà de 0.9 de corrélation c'est redondant. Ici rien ne dépasse ça, le maximum c'est `prise_ccs`/`prise_type2` à -0.82, ce qui s'explique juste par le fait qu'une borne a rarement les deux types de prise.

`prise_autre` a déjà été enlevée avant ce calcul (2.8% de présence, trop rare).

In [5]:
corr = df_propre[colonnes_numeriques + colonnes_booleennes].corr().round(2)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, annot_kws={'size': 7})
plt.title("Matrice de corrélation des variables retenues", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("output/matrice_correlation.png", dpi=300)
plt.close()


# 4. Prétraitement des données (Preprocessing)

## 4.1 Séparation du jeu de données (Train/Test Split)

`test_size=0.2`, `stratify=y` parce que les classes sont déséquilibrées.

On garde un split aléatoire classique (pas regroupé par station), même si on sait que ça pose un risque de fuite avec la lat/long (voir section 2 : 0.81 réel contre 0.95 avec ce split).

In [6]:
X = df_propre[colonnes_features].copy()
y = df_propre[colonne_cible]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


## 4.2 Normalisation des fonctionnalités et encodage OneHot (sauvegarde des deux)

On normalise (StandardScaler) seulement les colonnes numériques et booléennes, fit sur le train seulement. Pour `type_tarif` on applique le OneHotEncoder, fit sur le train aussi, avec `handle_unknown='ignore'` pour pas que ça plante si une catégorie inconnue apparaît plus tard. On sauvegarde les deux (joblib) parce que `main.py` doit refaire exactement la même transformation, sans jamais réajuster.

On concatène ensuite tout dans `X_train_scaled`/`X_test_scaled` pour que la suite du notebook n'ait pas à changer.

In [7]:
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[colonnes_numeriques + colonnes_booleennes])
X_test_num = scaler.transform(X_test[colonnes_numeriques + colonnes_booleennes])
joblib.dump(scaler, 'scaler_pretraitement_b3.pkl')

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_ohe = ohe.fit_transform(X_train[[colonne_categorielle]])
X_test_ohe = ohe.transform(X_test[[colonne_categorielle]])
joblib.dump(ohe, 'onehot_type_tarif_b3.pkl')

colonnes_ohe = list(ohe.get_feature_names_out([colonne_categorielle]))
colonnes_features_modele = colonnes_numeriques + colonnes_booleennes + colonnes_ohe
joblib.dump(colonnes_features_modele, 'feature_order_b3.pkl')

X_train_scaled = np.hstack([X_train_num, X_train_ohe])
X_test_scaled = np.hstack([X_test_num, X_test_ohe])

print(f"Nombre de colonnes finales du modèle : {len(colonnes_features_modele)}")
print(colonnes_features_modele)


Nombre de colonnes finales du modèle : 18
['puissance', 'nb_pdc', 'latitude', 'longitude', 'gratuit', 'deux_roues', 'prise_ccs', 'prise_type2', 'prise_chademo', 'prise_ef', 'paiement_acte', 'paiement_cb', 'paiement_autre', 'type_tarif_composite', 'type_tarif_gratuit', 'type_tarif_inconnu', 'type_tarif_kwh', 'type_tarif_temps']


# 5. Entraînement du modèle et optimisation (GridSearchCV)

On compare 4 algos : Régression Logistique, Random Forest, Gradient Boosting, K-Nearest Neighbors.

La régression logistique seule plafonne à 0.51 d'accuracy dans la version précédente — ses frontières linéaires captent pas les interactions entre puissance/nb_pdc/types de prise. Du coup on teste plusieurs familles de modèles avec GridSearchCV et on garde celui qui marche le mieux.

Petit point important sur la façon de choisir le meilleur modèle : on le choisit sur le score de validation croisée (`best_score_`, calculé sur les plis du train), pas sur l'accuracy du test. Si on choisissait sur le test puis qu'on rapportait ce même score, ça serait biaisé (on aurait pioché le meilleur des 4 sur le test, donc le chiffre serait un peu trop optimiste). Le test n'est utilisé qu'une fois, en section 6, sur le modèle déjà choisi.

On a aussi ajouté `class_weight: [None, 'balanced']` dans la grille de la régression logistique et du Random Forest, pour voir si rééquilibrer les classes aide (`Voirie` est largement majoritaire face à `Parking privé réservé à la clientèle` qui n'a que 307 individus). GradientBoosting et KNN ne gèrent pas ça nativement dans scikit-learn, on n'a pas cherché plus loin pour rester simple.

Comment marchent les 4 algos, en gros :

Régression Logistique : pour chaque classe, combinaison linéaire des variables passée dans une fonction softmax pour avoir une probabilité. Frontières de décision linéaires.

Random Forest : plein d'arbres de décision entraînés chacun sur un échantillon bootstrap et un sous-ensemble de variables, et on vote à la fin. Capture bien les interactions non linéaires.

Gradient Boosting : pareil mais les arbres sont construits les uns après les autres, chacun corrige les erreurs du précédent.

K-Nearest Neighbors : regarde les k voisins les plus proches et prend la classe majoritaire parmi eux.

In [8]:
resultats_algos = {}

print("Recherche des meilleurs hyperparamètres (Régression Logistique)...")
param_grid_lr = {'C': [0.1, 1.0, 10.0], 'solver': ['lbfgs', 'saga'], 'max_iter': [200], 'class_weight': [None, 'balanced']}
grid_lr = GridSearchCV(LogisticRegression(random_state=42), param_grid_lr, cv=5, scoring='accuracy', n_jobs=-1)
grid_lr.fit(X_train_scaled, y_train)
resultats_algos['Régression Logistique'] = {'modele': grid_lr.best_estimator_, 'params': grid_lr.best_params_, 'cv_score': grid_lr.best_score_}

print("Recherche des meilleurs hyperparamètres (Random Forest)...")
param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [10, 20, None], 'min_samples_leaf': [1, 5], 'class_weight': [None, 'balanced']}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train_scaled, y_train)
resultats_algos['Random Forest'] = {'modele': grid_rf.best_estimator_, 'params': grid_rf.best_params_, 'cv_score': grid_rf.best_score_}

print("Recherche des meilleurs hyperparamètres (Gradient Boosting)...")
param_grid_gb = {'n_estimators': [100], 'learning_rate': [0.1, 0.2], 'max_depth': [3, 5]}
grid_gb = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid_gb, cv=5, scoring='accuracy', n_jobs=-1)
grid_gb.fit(X_train_scaled, y_train)
resultats_algos['Gradient Boosting'] = {'modele': grid_gb.best_estimator_, 'params': grid_gb.best_params_, 'cv_score': grid_gb.best_score_}

print("Recherche des meilleurs hyperparamètres (K-Nearest Neighbors)...")
param_grid_knn = {'n_neighbors': [5, 9, 15], 'weights': ['uniform', 'distance']}
grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train_scaled, y_train)
resultats_algos['K-Nearest Neighbors'] = {'modele': grid_knn.best_estimator_, 'params': grid_knn.best_params_, 'cv_score': grid_knn.best_score_}

print("\nCOMPARAISON DES MODÈLES (accuracy en validation croisée, jeu d'entraînement uniquement) :")
print('=' * 75)
for nom, info in resultats_algos.items():
    print(f"{nom:25s} -> cv_score = {info['cv_score']:.4f}  | params = {info['params']}")

meilleur_nom = max(resultats_algos, key=lambda n: resultats_algos[n]['cv_score'])
meilleur_modele = resultats_algos[meilleur_nom]['modele']
print(f"\nModèle retenu (sur score CV, jeu de test pas encore consulté) : {meilleur_nom} "
      f"(cv_score = {resultats_algos[meilleur_nom]['cv_score']:.4f})")


Recherche des meilleurs hyperparamètres (Régression Logistique)...
Recherche des meilleurs hyperparamètres (Random Forest)...
Recherche des meilleurs hyperparamètres (Gradient Boosting)...
Recherche des meilleurs hyperparamètres (K-Nearest Neighbors)...

COMPARAISON DES MODÈLES (accuracy en validation croisée, jeu d'entraînement uniquement) :
Régression Logistique     -> cv_score = 0.6873  | params = {'C': 10.0, 'class_weight': None, 'max_iter': 200, 'solver': 'lbfgs'}
Random Forest             -> cv_score = 0.9612  | params = {'class_weight': None, 'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Gradient Boosting         -> cv_score = 0.9075  | params = {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100}
K-Nearest Neighbors       -> cv_score = 0.9452  | params = {'n_neighbors': 15, 'weights': 'distance'}

Modèle retenu (sur score CV, jeu de test pas encore consulté) : Random Forest (cv_score = 0.9612)


# 6. Évaluation du modèle final

On utilise `classification_report` (précision/rappel/f1 par classe) plutôt que juste l'accuracy globale, parce que les classes sont déséquilibrées et l'accuracy seule cacherait une mauvaise prédiction sur les classes minoritaires.

In [9]:
y_pred = meilleur_modele.predict(X_test_scaled)

rapport_dict = classification_report(y_test, y_pred, output_dict=True)
rapport_df = pd.DataFrame(rapport_dict).transpose().round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(rapport_df[['precision', 'recall', 'f1-score']], annot=True, cmap='YlGnBu', vmin=0, vmax=1)
plt.title(f"Rapport de classification - {meilleur_nom}", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("output/rapport_classification.png", dpi=300)
plt.close()


## 6.1 Matrice de confusion

Ça complète le rapport de classification en montrant concrètement quelles classes sont confondues entre elles — on voit notamment que `Parking public` se confond beaucoup avec `Voirie` (détails dans la discussion plus bas).

In [10]:
matrice = confusion_matrix(y_test, y_pred)
classes = meilleur_modele.classes_

plt.figure(figsize=(8, 6))
sns.heatmap(matrice, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Matrice de Confusion - Implantation de la Station", fontsize=12, fontweight='bold')
plt.ylabel('Réalité Terrain')
plt.xlabel('Prédiction Modèle')
plt.tight_layout()
plt.savefig("output/matrice_confusion.png", dpi=300)
plt.close()

## 6.1bis Analyse des Vrais/Faux Positifs et Négatifs

Pour chaque classe (en regardant cette classe contre toutes les autres) :
- Vrai Positif (VP) : le modèle dit cette classe, et c'est la bonne.
- Faux Négatif (FN) : c'est vraiment cette classe, mais le modèle en a prédit une autre (raté).
- Faux Positif (FP) : le modèle dit cette classe, mais en vrai c'est une autre (fausse alerte).
- Vrai Négatif (VN) : ni la réalité ni la prédiction ne sont cette classe.

On rajoute ça parce que précision et rappel sont des ratios, ça ne dit pas combien d'erreurs il y a vraiment en nombre brut. Le graphique en dessous montre les comptes réels par classe.

In [11]:
vrais_positifs = np.diag(matrice)
faux_negatifs = matrice.sum(axis=1) - vrais_positifs
faux_positifs = matrice.sum(axis=0) - vrais_positifs
total = matrice.sum()
vrais_negatifs = total - (vrais_positifs + faux_negatifs + faux_positifs)

tableau_erreurs = pd.DataFrame({
    'Vrais Positifs': vrais_positifs,
    'Faux Negatifs': faux_negatifs,
    'Faux Positifs': faux_positifs,
    'Vrais Negatifs': vrais_negatifs,
}, index=classes)

tableau_erreurs[['Vrais Positifs', 'Faux Negatifs', 'Faux Positifs']].plot(
    kind='bar', figsize=(10, 6), color=['seagreen', 'indianred', 'goldenrod']
)
plt.title("Vrais Positifs / Faux Negatifs / Faux Positifs par classe", fontsize=12, fontweight='bold')
plt.ylabel("Nombre d'observations")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig("output/tp_fn_fp_par_classe.png", dpi=300)
plt.close()


## 6.1ter Importance des variables (Feature Importance)

On vérifie ici que les variables qu'on a gardées (colonnes OneHot de `type_tarif` incluses) servent vraiment au modèle, plutôt que de juste se fier à notre intuition de départ.

In [12]:
if hasattr(meilleur_modele, 'feature_importances_'):
    importances = pd.Series(meilleur_modele.feature_importances_, index=colonnes_features_modele).sort_values(ascending=False)
    titre = f"Importance des variables - {meilleur_nom}"
elif hasattr(meilleur_modele, 'coef_'):
    importances = pd.Series(np.abs(meilleur_modele.coef_).mean(axis=0), index=colonnes_features_modele).sort_values(ascending=False)
    titre = f"Importance des variables (|coefficients| moyens) - {meilleur_nom}"
else:
    importances = None

if importances is not None:
    plt.figure(figsize=(9, 6))
    sns.barplot(x=importances.values, y=importances.index, hue=importances.index, legend=False, palette='viridis')
    plt.title(titre, fontsize=12, fontweight='bold')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.savefig("output/feature_importance.png", dpi=300)
    plt.close()
else:
    print(f"{meilleur_nom} n'expose ni feature_importances_ ni coef_ (ex. KNN) - pas d'importance disponible.")


## 6.2 Vérification anti-fuite : split groupé par station

Le split aléatoire de la section 4.1 mélange les lignes au hasard sans regarder si une même station (en moyenne 3.3 points de charge) se retrouve des deux côtés. Avec la lat/long comme variables, le modèle peut reconnaître une station déjà vue à l'entraînement au lieu de vraiment généraliser — donc on vérifie ça ici.

On utilise `GroupShuffleSplit` en regroupant par `id_station`, comme ça chaque station est entièrement d'un côté ou de l'autre, jamais les deux. On reprend le même algo et les mêmes hyperparamètres que le modèle retenu, juste pour comparer — ce modèle-là n'est pas celui qu'on exporte en section 7, qui reste basé sur le split aléatoire de base.

In [13]:
groupes = df_propre['id_station']

separateur_groupe = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
indices_train, indices_test = next(separateur_groupe.split(df_propre, y, groups=groupes))

X_train_grp = df_propre.iloc[indices_train][colonnes_features]
X_test_grp = df_propre.iloc[indices_test][colonnes_features]
y_train_grp = df_propre.iloc[indices_train][colonne_cible]
y_test_grp = df_propre.iloc[indices_test][colonne_cible]

scaler_grp = StandardScaler()
X_train_grp_num = scaler_grp.fit_transform(X_train_grp[colonnes_numeriques + colonnes_booleennes])
X_test_grp_num = scaler_grp.transform(X_test_grp[colonnes_numeriques + colonnes_booleennes])

ohe_grp = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_grp_ohe = ohe_grp.fit_transform(X_train_grp[[colonne_categorielle]])
X_test_grp_ohe = ohe_grp.transform(X_test_grp[[colonne_categorielle]])

X_train_grp_scaled = np.hstack([X_train_grp_num, X_train_grp_ohe])
X_test_grp_scaled = np.hstack([X_test_grp_num, X_test_grp_ohe])

modele_grp = RandomForestClassifier(**meilleur_modele.get_params())
modele_grp.fit(X_train_grp_scaled, y_train_grp)
y_pred_grp = modele_grp.predict(X_test_grp_scaled)

accuracy_split_aleatoire = accuracy_score(y_test, y_pred)
accuracy_split_groupe = accuracy_score(y_test_grp, y_pred_grp)

comparaison_splits = pd.Series({
    'Split aleatoire (avec risque de fuite)': accuracy_split_aleatoire,
    'Split groupe par station (sans fuite)': accuracy_split_groupe,
})

comparaison_splits.plot(kind='bar', figsize=(7, 5), color=['indianred', 'seagreen'])
plt.title("Accuracy selon la methode de split", fontsize=12, fontweight='bold')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig("output/comparaison_splits.png", dpi=300)
plt.close()


## 6.3 Discussion des résultats

On présente les deux résultats séparément, parce que ça n'a pas de sens de les mélanger — c'est pas le même modèle, pas le même split, donc pas la même histoire.

**Résultat avec le split aléatoire (celui qu'on garde par défaut, section 4.1) :**

Accuracy sur le test : 0.97, cv_score en entraînement : 0.9615 (Random Forest). Avec ce modèle, toutes les classes dépassent 0.92 de rappel, même la classe rare `Parking privé réservé à la clientèle` (0.94, contre 0.52 dans la version précédente sans lat/long). C'est le modèle exporté en section 7, celui que `main.py` utilise.

Comparaison des 4 algos sur ce split (score de validation croisée) :

| Algorithme | CV score |
|---|---|
| Régression Logistique | 0.6873 |
| Random Forest | 0.9615 |
| Gradient Boosting | 0.9076 |
| K-Nearest Neighbors | 0.9452 |

**Résultat avec le split groupé par station (vérification anti-fuite, section 6.2) :**

Même algo, mêmes hyperparamètres, mais entraîné et testé sur des stations totalement séparées entre train et test. Accuracy : 0.81 (voir le graphique de comparaison de la section 6.2). Pas de détail par classe ici, on n'a calculé que l'accuracy globale pour ce modèle de vérification, il sert juste à comparer.

**Ce que ça veut dire :**

L'écart entre les deux (0.81 vs 0.97) vient presque entièrement de la fuite via lat/long : sur le split aléatoire, le modèle peut retomber sur une station déjà vue à l'entraînement (même station, juste un autre point de charge), donc il "reconnaît" plutôt que de généraliser. Le 0.81 du split par station, c'est le vrai signal géographique, sans tricherie. Les variables de paiement ajoutent aussi un raisonnement circulaire qu'on avait déjà écarté avant, donc même le 0.81 a un biais résiduel.

Si on présente le 0.97 comme résultat final, il faut vraiment expliquer ce compromis en soutenance, sinon le chiffre donne une fausse impression. Pour un résultat honnête sur une borne vraiment nouvelle, le modèle de la version précédente (0.76, sans lat/long ni paiement, sans risque de fuite) reste la référence la plus solide.

# 7. Exportation du modèle prédictif final

On sauvegarde le modèle (joblib) pour que `main.py` puisse le charger directement sans jamais relancer l'entraînement.

In [14]:
joblib.dump(meilleur_modele, 'modele_classification_b3.pkl')


['modele_classification_b3.pkl']